## WIDGETS

In [0]:
dbutils.widgets.text("catalog","ecommerce_catalog_dev")
catalog = dbutils.widgets.get("catalog")

## Daily Performance 

In [0]:
%sql
-- 1. Daily Performance KPI Table
CREATE OR REPLACE VIEW IDENTIFIER(:catalog).gold.daily_revenue_kpi AS
SELECT 
    order_date,
    ROUND(SUM(total_amount), 2) AS total_daily_revenue,
    COUNT(DISTINCT order_id) AS total_completed_orders,
    COUNT(DISTINCT customer_id) AS distinct_active_customers
FROM IDENTIFIER(:catalog).silver.silver_table
WHERE status = 'COMPLETED'
GROUP BY order_date
ORDER BY order_date DESC;

## Product Metrics 

In [0]:
%sql
-- 2. Product Metrics KPI Table
CREATE OR REPLACE VIEW IDENTIFIER(:catalog).gold.product_performance_kpi AS
SELECT 
    product_name,
    SUM(quantity) AS total_units_sold,
    ROUND(SUM(total_amount), 2) AS gross_revenue,
    ROUND(AVG(unit_price), 2) AS avg_selling_price
FROM IDENTIFIER(:catalog).silver.silver_table
WHERE status = 'COMPLETED'
GROUP BY product_name
ORDER BY gross_revenue DESC;

## Customer Value

In [0]:
%sql
-- 3. Customer Value (CLV) KPI Table
CREATE OR REPLACE VIEW IDENTIFIER(:catalog).gold.customer_clv_kpi AS
SELECT 
    customer_id,
    COUNT(DISTINCT order_id) AS total_completed_orders,
    ROUND(SUM(total_amount), 2) AS customer_lifetime_spend,
    ROUND(AVG(total_amount), 2) AS avg_order_value
FROM IDENTIFIER(:catalog).silver.silver_table
WHERE status = 'COMPLETED'
GROUP BY customer_id
ORDER BY customer_lifetime_spend DESC;